# Blockchain & Cryptography: Combo Nature System
### Comprehensive End-to-End Cryptographic Ledger Implementation

This notebook implements the complete **Blockchain and Cryptography Combo Nature System** based on the technical specifications:
1. **Cryptographic Hash Functions**: SHA-256 hashing, avalanche effect, and bitwise sensitivity.
2. **Asymmetric Key Cryptography**: ECDSA (SECP256k1) public/private key generation, message signing, and signature verification.
3. **Merkle Trees**: Cryptographic transaction tree generation, Merkle root calculation, and SPV inclusion proofs.
4. **Transactions & Mempool**: Peer-to-peer transfers (Vinny and Kinny), digital signatures, balance enforcement, and double-spend prevention.
5. **Block Construction & Proof of Work**: Mining engine, difficulty targets, and nonce search.
6. **Blockchain Ledger**: Genesis block, state accounting, chain validation, and tamper-resistance verification.


## Step 1: Environment Setup & Dependencies
We utilize Python standard libraries (`hashlib`, `json`, `time`, `random`) alongside the standard `cryptography` library for SECP256k1 elliptic curve operations.

In [1]:
# Install cryptography if running in fresh environment
!pip install -q cryptography

import hashlib
import json
import time
import random
from typing import List, Dict, Optional, Tuple, Any
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.exceptions import InvalidSignature

print('Environment initialized successfully!')


Environment initialized successfully!


## Step 2: SHA-256 Hashing and the Avalanche Effect
As described in the project synopsis, a cryptographic hash function provides:
- **Deterministic**: Same input always yields the exact same 256-bit output.
- **One-Way Pre-Image Resistance**: Impossible to reverse-engineer input from the hash.
- **Avalanche Effect**: A 1-character difference in the input produces a totally unrelated output (~50% of the 256 bits flip).


In [2]:
def sha256_hex(data: Any) -> str:
    '''Compute SHA-256 64-character hexadecimal hash.'''
    if isinstance(data, (dict, list)):
        payload = json.dumps(data, sort_keys=True, separators=(',', ':')).encode('utf-8')
    elif isinstance(data, str):
        payload = data.encode('utf-8')
    elif isinstance(data, bytes):
        payload = data
    else:
        payload = str(data).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

def sha256_binary(data: Any) -> str:
    '''Convert SHA-256 hash to a 256-character string of 0s and 1s.'''
    h = sha256_hex(data)
    return ''.join(f'{int(h[i:i+2], 16):08b}' for i in range(0, len(h), 2))

# Test example from Synopsis: 'Blockchain at myfort' vs 'Blockchain at Myfort'
input1 = 'Blockchain at myfort'
input2 = 'Blockchain at Myfort'

hash1 = sha256_hex(input1)
hash2 = sha256_hex(input2)

bin1 = sha256_binary(input1)
bin2 = sha256_binary(input2)

differing_bits = sum(b1 != b2 for b1, b2 in zip(bin1, bin2))
divergence_pct = (differing_bits / 256) * 100

print(f'Input 1: {input1}')
print(f'Hash 1:  {hash1}')
print(f'\nInput 2: {input2}')
print(f'Hash 2:  {hash2}')
print(f'\n[AVALANCHE EFFECT]: {differing_bits} out of 256 bits changed ({divergence_pct:.2f}% divergence)!')


Input 1: Blockchain at myfort
Hash 1:  793159ad5f30610165aaea175938e9303400ae05a668fc8910f5051ec153cbfe

Input 2: Blockchain at Myfort
Hash 2:  a9ee314c4ec9f2a0ada42f1c0a5752e9150453f061fb5c8608f29e7976518913

[AVALANCHE EFFECT]: 127 out of 256 bits changed (49.61% divergence)!


## Step 3: Asymmetric Cryptography (ECDSA SECP256k1)
Digital signatures guarantee:
- **Integrity**: Transaction cannot be altered in transit.
- **Authentication**: Proof that the sender owns the private key.
- **Non-Repudiation**: The sender cannot deny having authorized the transaction.


In [3]:
class Wallet:
    def __init__(self, name: str = 'Wallet'):
        self.name = name
        self.private_key = ec.generate_private_key(ec.SECP256K1())
        self.public_key = self.private_key.public_key()
        
        self.priv_pem = self.private_key.private_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PrivateFormat.PKCS8,
            encryption_algorithm=serialization.NoEncryption()
        ).decode('utf-8')
        
        self.pub_pem = self.public_key.public_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PublicFormat.SubjectPublicKeyInfo
        ).decode('utf-8')
        
        uncompressed = self.public_key.public_bytes(
            encoding=serialization.Encoding.X962,
            format=serialization.PublicFormat.UncompressedPoint
        )
        self.address = f'0x{hashlib.sha256(uncompressed).hexdigest()[:40]}'

    def sign(self, message: str) -> str:
        sig = self.private_key.sign(message.encode('utf-8'), ec.ECDSA(hashes.SHA256()))
        return sig.hex()

    @staticmethod
    def verify(pub_pem: str, signature_hex: str, message: str) -> bool:
        try:
            pk = serialization.load_pem_public_key(pub_pem.encode('utf-8'))
            pk.verify(bytes.fromhex(signature_hex), message.encode('utf-8'), ec.ECDSA(hashes.SHA256()))
            return True
        except Exception:
            return False

# Create wallets for Vinny and Kinny
vinny = Wallet('Vinny')
kinny = Wallet('Kinny')
print(f'Vinny Address: {vinny.address}')
print(f'Kinny Address: {kinny.address}')

test_msg = 'Pay Kinny 15 coins'
sig = vinny.sign(test_msg)
print(f'Digital Signature: {sig[:32]}...')
print(f'Signature Verification: {Wallet.verify(vinny.pub_pem, sig, test_msg)}')
print(f'Tampered Message Check: {Wallet.verify(vinny.pub_pem, sig, "Pay Kinny 500 coins")}')


Vinny Address: 0xcd0fdb4e106a3c58f243f505c759016c9ad2edb2
Kinny Address: 0xbb2df3436a9356d12c741e792c725b2fad6cb6fe
Digital Signature: 304402203093a76d76dd2a373e8538d8...
Signature Verification: True
Tampered Message Check: False


## Step 4: Merkle Tree Implementation
A binary Merkle tree hierarchically hashes pairs of transaction hashes up to a single **Merkle Root**, enabling fast verification and inclusion proofs.

In [4]:
class MerkleTree:
    def __init__(self, leaves: List[str]):
        self.leaves = leaves
        self.levels: List[List[str]] = []
        self.root = self.build_tree()
        
    def build_tree(self) -> str:
        if not self.leaves:
            return '0' * 64
        current_level = [sha256_hex(leaf) if len(leaf) != 64 else leaf for leaf in self.leaves]
        self.levels = [current_level]
        while len(current_level) > 1:
            if len(current_level) % 2 != 0:
                current_level.append(current_level[-1])
            next_level = []
            for i in range(0, len(current_level), 2):
                parent = sha256_hex(current_level[i] + current_level[i+1])
                next_level.append(parent)
            current_level = next_level
            self.levels.append(current_level)
        return self.levels[-1][0]

tree = MerkleTree(['tx1_data', 'tx2_data', 'tx3_data'])
print(f'Merkle Root: {tree.root}')
print(f'Tree Levels: {len(tree.levels)}')


Merkle Root: e441a86973ca334cf6e758a7ee39971510c0621a5ab83bacab1c2564b3d853f4
Tree Levels: 3


## Step 5: Transaction Structure & Digital Signing
Transactions represent signed transfers of tokens between accounts. Each transaction carries the sender public key, signature, and is checked for validity.

In [5]:
class Transaction:
    def __init__(self, sender: str, recipient: str, amount: float, sender_public_key: Optional[str] = None):
        self.sender = sender
        self.recipient = recipient
        self.amount = round(float(amount), 8)
        self.timestamp = time.time()
        self.sender_public_key = sender_public_key
        self.signature: Optional[str] = None
        self.tx_id = self.compute_hash()

    def get_payload(self) -> str:
        d = {'sender': self.sender, 'recipient': self.recipient, 'amount': self.amount, 'timestamp': round(self.timestamp, 4)}
        return sha256_hex(d)

    def compute_hash(self) -> str:
        return self.get_payload()

    def sign(self, wallet: Wallet):
        if self.sender == 'SYSTEM':
            self.signature = 'COINBASE_SYSTEM'
            return
        self.signature = wallet.sign(self.get_payload())

    def is_valid(self) -> bool:
        if self.amount <= 0:
            return False
        if self.sender == 'SYSTEM':
            return True
        if not self.signature or not self.sender_public_key:
            return False
        return Wallet.verify(self.sender_public_key, self.signature, self.get_payload())

    def to_dict(self) -> dict:
        return {
            'tx_id': self.tx_id, 'sender': self.sender, 'recipient': self.recipient,
            'amount': self.amount, 'timestamp': self.timestamp, 'signature': self.signature
        }

tx = Transaction(sender=vinny.address, recipient=kinny.address, amount=25.0, sender_public_key=vinny.pub_pem)
tx.sign(vinny)
print(f'Created and signed transaction: {tx.tx_id[:16]}... Valid? {tx.is_valid()}')


Created and signed transaction: ffaf42c81b8aba55... Valid? True


## Step 6: Block Structure & Proof of Work (PoW) Mining
Each block encapsulates:
- `index`, `timestamp`, `previous_hash`, `merkle_root`, `nonce`, `difficulty`
- Proof-of-Work loop: search for a nonce such that SHA256(BlockHeader) starts with D leading zeros.

In [6]:
class Block:
    def __init__(self, index: int, transactions: List[Transaction], previous_hash: str, difficulty: int = 2):
        self.index = index
        self.transactions = transactions
        self.previous_hash = previous_hash
        self.difficulty = difficulty
        self.timestamp = time.time()
        self.nonce = 0
        self.merkle_root = self.compute_merkle_root()
        self.hash = self.compute_hash()

    def compute_merkle_root(self) -> str:
        tx_hashes = [tx.tx_id for tx in self.transactions]
        return MerkleTree(tx_hashes).root

    def compute_hash(self) -> str:
        header = {
            'index': self.index,
            'timestamp': round(self.timestamp, 4),
            'previous_hash': self.previous_hash,
            'merkle_root': self.merkle_root,
            'nonce': self.nonce,
            'difficulty': self.difficulty
        }
        return sha256_hex(header)

    def mine(self) -> Tuple[str, int]:
        target = '0' * self.difficulty
        while True:
            h = self.compute_hash()
            if h.startswith(target):
                self.hash = h
                return h, self.nonce
            self.nonce += 1

    def is_valid(self, expected_previous_hash: Optional[str] = None) -> bool:
        if expected_previous_hash and self.previous_hash != expected_previous_hash:
            return False
        if self.compute_merkle_root() != self.merkle_root:
            return False
        if self.compute_hash() != self.hash:
            return False
        if not self.hash.startswith('0' * self.difficulty):
            return False
        return all(tx.is_valid() for tx in self.transactions)

b = Block(1, [tx], '0'*64, difficulty=2)
h, n = b.mine()
print(f'Block Mined: Hash = {h}, Nonce = {n}, Valid = {b.is_valid("0"*64)}')


Block Mined: Hash = 00d9e79e8606dacd83a4cbbb8c206f83d3a9ea16f30abf27715e7319e8e11cd3, Nonce = 120, Valid = True


## Step 7: Blockchain Ledger & Consensus Engine
The `Blockchain` maintains the chain of blocks, ledger balances, transaction mempool, and validation rules.

In [7]:
class Blockchain:
    def __init__(self, difficulty: int = 2, reward: float = 25.0):
        self.difficulty = difficulty
        self.reward = reward
        self.chain: List[Block] = []
        self.pending_txs: List[Transaction] = []
        self.create_genesis_block()

    def create_genesis_block(self):
        genesis_tx = Transaction(sender='SYSTEM', recipient='0xGenesisFaucet', amount=1000.0)
        genesis_tx.signature = 'GENESIS'
        genesis_block = Block(0, [genesis_tx], '0' * 64, difficulty=self.difficulty)
        genesis_block.mine()
        self.chain.append(genesis_block)

    def get_latest_block(self) -> Block:
        return self.chain[-1]

    def get_balance(self, address: str) -> float:
        bal = 0.0
        for block in self.chain:
            for tx in block.transactions:
                if tx.recipient == address:
                    bal += tx.amount
                if tx.sender == address:
                    bal -= tx.amount
        for tx in self.pending_txs:
            if tx.sender == address:
                bal -= tx.amount
        return round(bal, 4)

    def add_transaction(self, tx: Transaction) -> bool:
        if not tx.is_valid():
            print(f'Transaction {tx.tx_id[:8]} rejected: Invalid signature.')
            return False
        if tx.sender != 'SYSTEM' and self.get_balance(tx.sender) < tx.amount:
            print(f'Transaction {tx.tx_id[:8]} rejected: Insufficient balance.')
            return False
        self.pending_txs.append(tx)
        return True

    def mine_pending(self, miner_address: str) -> Block:
        reward_tx = Transaction('SYSTEM', miner_address, self.reward)
        reward_tx.signature = 'REWARD'
        block_txs = [reward_tx] + self.pending_txs
        new_block = Block(len(self.chain), block_txs, self.get_latest_block().hash, self.difficulty)
        new_block.mine()
        self.chain.append(new_block)
        self.pending_txs = []
        return new_block

    def is_valid(self) -> Tuple[bool, str]:
        for i in range(1, len(self.chain)):
            curr = self.chain[i]
            prev = self.chain[i-1]
            if curr.previous_hash != prev.hash:
                return False, f'Block #{i} previous_hash link broken!'
            if not curr.is_valid(prev.hash):
                return False, f'Block #{i} internal validation failed!'
        return True, 'Blockchain is completely valid and secure.'

print('Blockchain class ready!')


Blockchain class ready!


## Step 8: Multi-Party Simulation & Token Conservation
Simulating exchanges between Vinny, Kinny, and Miner, tracking balances and verifying consensus.

In [8]:
# Initialize fresh blockchain
blockchain = Blockchain(difficulty=2, reward=50.0)
miner = Wallet('Miner')

# 1. Miner mines block to bootstrap network
blockchain.mine_pending(miner.address)
print(f'Miner balance after block 1: {blockchain.get_balance(miner.address)}')

# 2. Miner sends 30 tokens to Vinny
tx1 = Transaction(miner.address, vinny.address, 30.0, miner.pub_pem)
tx1.sign(miner)
blockchain.add_transaction(tx1)

# 3. Mine block 2
blockchain.mine_pending(miner.address)
print(f'Vinny balance after confirmation: {blockchain.get_balance(vinny.address)}')

# 4. Vinny transfers 12 tokens to Kinny
tx2 = Transaction(vinny.address, kinny.address, 12.0, vinny.pub_pem)
tx2.sign(vinny)
blockchain.add_transaction(tx2)
blockchain.mine_pending(miner.address)

print(f'Vinny final balance: {blockchain.get_balance(vinny.address)}')
print(f'Kinny final balance: {blockchain.get_balance(kinny.address)}')
print(f'Miner final balance: {blockchain.get_balance(miner.address)}')

valid, msg = blockchain.is_valid()
print(f'Blockchain Status: {msg} (Valid: {valid})')


Miner balance after block 1: 50.0
Vinny balance after confirmation: 30.0
Vinny final balance: 18.0
Kinny final balance: 12.0
Miner final balance: 120.0
Blockchain Status: Blockchain is completely valid and secure. (Valid: True)


## Step 9: Tamper Resistance & Avalanche Demonstration
We deliberately modify a historical transaction inside Block #1. We demonstrate how the cryptographic properties of the block and the avalanche effect immediately detect the alteration and break chain consensus.

In [9]:
print('--- TAMPER EXPERIMENT ---')
target_block = blockchain.chain[1]
original_amount = target_block.transactions[0].amount

print(f'Original Block #1 TX amount: {original_amount}')
# Adversary alters amount to 999999
target_block.transactions[0].amount = 999999.0
target_block.transactions[0].tx_id = target_block.transactions[0].compute_hash()

valid, msg = blockchain.is_valid()
print(f'After Tamper Attack -> Is Chain Valid? {valid}')
print(f'Consensus Diagnostic: {msg}')
assert valid == False, 'Security failure: tampering went undetected!'
print('\nSUCCESS: The cryptographic blockchain successfully defended against data corruption!')


--- TAMPER EXPERIMENT ---
Original Block #1 TX amount: 50.0
After Tamper Attack -> Is Chain Valid? False
Consensus Diagnostic: Block #1 internal validation failed!

SUCCESS: The cryptographic blockchain successfully defended against data corruption!
